In [1]:
from importlib import reload
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd
import numpy as np
import pyautogui
import os

from adcp_toolbox import select, plot

# Plot data from selected files

### Select files

In [2]:
filepaths = select.multiple_files()

filepaths

('C:/Users/Nortek1000/Desktop/toolbox/data/netcdf/hval_2025-10-11_37_days.nc',)

### Specify plot parameters

In [153]:

y_mode = 'range'
ylim = (22,0)
clim = 1.2

wrap_roll = True # use if ADCP is mounted upside-down

corr_threshold = 40

datefmt='%H:%M'


plot_types = [
#    'velocity',
    'amplitude',
#    'correlation',
#    'orientation+PT'
]

out_directory = '../../adcp-data/figures/2025-10'

### Plot data

In [148]:
def get_date_title(ds):
    # First/last timestamps
    time0 = pd.to_datetime(ds['TIME'].values[0])
    time1 = pd.to_datetime(ds['TIME'].values[-1])
    
    # Duration
    duration = time1 - time0
    total_minutes = int(duration.total_seconds() // 60)
    
    days, rem = divmod(total_minutes, 24 * 60)
    hours, minutes = divmod(rem, 60)
    
    if days > 0:
        duration_str = f'{days}d {hours}h {minutes}min'
    elif hours > 0:
        duration_str = f'{hours}h {minutes}min'
    else:
        duration_str = f'{minutes} min'
    
    # Main title
    date_str = time0.strftime('%Y-%m-%d %H:%M:%S')
    title = f'{date_str}  -  {duration_str}'

    return title


In [154]:
reload(plot)

for filepath in filepaths:
    ds = xr.open_dataset(filepath)

    for plot_type in plot_types:

        fig = plot.plot_data(
            ds,
            plot_type=plot_type,
            corr_threshold=corr_threshold if plot_type == 'velocity' else None,
            clim=clim if plot_type == 'velocity' else None,
            ylim=ylim if plot_type != 'orientation+PT' else None,
            y_mode=y_mode,
            datefmt=datefmt,
            wrap_roll=wrap_roll,
            grid=True,
        )
        
        # Title and filename
        title = get_date_title(ds)
        filename = filepath.rpartition('/')[-1]
        filename_wo_ext = filename.split('.')[0]
        
        # Titles
        title_offset = 0.56
        filename_offset = 0.28
        
        fig_height = fig.get_size_inches()[1]
        subplot_top = max(ax.get_position().y1 for ax in fig.axes)
        ytitle = subplot_top + title_offset / fig_height
        yfilename = ytitle - filename_offset / fig_height
        
        fig.suptitle(title, fontsize=15, y=ytitle)
        fig.text(0.5, yfilename, filename, ha='center', va='top', fontsize=10)
        
        # Create output directory
        plot_directory = os.path.join(out_directory, plot_type)
        os.makedirs(plot_directory, exist_ok=True)

        display(fig)
        
        # Save plot
        plot_filename = f'{filename_wo_ext}_{plot_type}.png'
        plot_path = os.path.join(plot_directory, plot_filename)
        #fig.savefig(plot_path, bbox_inches='tight')

        # Print out confirmation message
        #print('Saved', plot_filename)
    
    ds.close()

KeyError: "No variable named 'A5'. Variables on the dataset include ['CP', 'H', 'PI', 'R', 'P', ..., 'A2', 'A3', 'A4', 'TIME', 'CN']"